In [3]:
#  Loading env files

from dotenv import load_dotenv
load_dotenv()

True

In [4]:

import os
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import HumanMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


In [5]:

model = ChatGroq(model="llama-3.1-8b-instant" , groq_api_key= os.getenv("GROQ_API_KEY"))
store={}

def get_session_history(session_id : str)-> BaseChatMessageHistory: 
    if session_id not in store: 
        store[session_id] = ChatMessageHistory()
    return store[session_id]
with_message_history=RunnableWithMessageHistory(model , get_session_history)


In [6]:
config= {"configurable" : {"session_id" : "chat1"}}
response =with_message_history.invoke([HumanMessage(content= "Hi , my name is ahad and i am a developing software engineer")] , config= config)
response.content

"Nice to meet you, Ahad. As a developing software engineer, you're likely working on various projects and technologies. What area of software development are you most interested in or currently focusing on (e.g., web development, mobile app development, machine learning, AI, etc.)?"

In [7]:
response =with_message_history.invoke([HumanMessage(content= "what is my name")] , config= config)
response.content

'Your name is Ahad.'

In [50]:
# Chaning the config 

config3= {"configurable" : {"session_id" : "chat1"}}
response =with_message_history.invoke([HumanMessage(content= "my name is ajo")] , config= config3)
response.content

'Nice to meet you, Ajo. Is there something I can help you with today?'

In [8]:
#  Prompt Tempelate 
from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder

prompt= ChatPromptTemplate (
    [
        ("system" , "You're a helpful assistant , answer all the questions to the best of your ability" ), 
        MessagesPlaceholder(variable_name= "messages")
    ]
)


In [10]:
chain = prompt | model
chain.invoke({"messages" :[ HumanMessage(content = "Hii my name is jack")] })

AIMessage(content='Nice to meet you, Jack. Is there something I can help you with today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 57, 'total_tokens': 75, 'completion_time': 0.029567273, 'completion_tokens_details': None, 'prompt_time': 0.002803173, 'prompt_tokens_details': None, 'queue_time': 0.051931644, 'total_time': 0.032370446}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dec53-a012-7d71-b46d-918d2b415555-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 18, 'total_tokens': 75})

In [11]:
with_message_history = RunnableWithMessageHistory(chain , get_session_history)

In [37]:
config={"configurable" :{"session_id" : "chat3"}}
response = with_message_history.invoke([HumanMessage(content = "Hii my name is jack")], config= config)
response

AIMessage(content="Nice to meet you, Jack. It looks like you just introduced yourself again. Is everything okay? Would you like to chat or is there something specific you'd like to talk about?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 87, 'total_tokens': 125, 'completion_time': 0.074584323, 'completion_tokens_details': None, 'prompt_time': 0.004861101, 'prompt_tokens_details': None, 'queue_time': 0.051924775, 'total_time': 0.079445424}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019de7cb-9525-7ef3-9f85-0865aa82498e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 38, 'total_tokens': 125})

In [12]:
# Adding multiple keys

from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder

prompt= ChatPromptTemplate (
    [
        ("system" , "You're a helpful assistant , answer all the questions to the best of your ability in {language}" ), 
        MessagesPlaceholder(variable_name= "messages")
    ]
)

In [13]:
chain = prompt | model
chain.invoke({"messages" :[ HumanMessage(content = "Hii my name is jack")] , "language" : "hindi" })

AIMessage(content='नमस्ते जैक! मैं आपकी सहायता करने के लिए यहाँ हूँ। क्या आपको कोई विशेष समस्या या प्रश्न है जिसके लिए मैं आपकी मदद कर सकता हूँ?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 59, 'total_tokens': 122, 'completion_time': 0.089769759, 'completion_tokens_details': None, 'prompt_time': 0.005457548, 'prompt_tokens_details': None, 'queue_time': 0.057592852, 'total_time': 0.095227307}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dec53-b882-7431-a713-656b5fa12846-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 63, 'total_tokens': 122})

In [17]:
from langchain_core.messages import SystemMessage , trim_messages
from langchain_core.messages import AIMessage

trimmer = trim_messages(
    max_tokens= 10,
    strategy= "last" , 
    token_counter= model,
    include_system= True,
    allow_partial= True,
    start_on= "human"
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

In [18]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(message=itemgetter("messages") | trimmer)
    | prompt 
    | model
)

response = chain.invoke(
    {
        "messages" : messages + [HumanMessage(content= "what is my name")],
        "language" : "English"
    }
)
response.content

'your name is bob'

In [19]:
#  Wrapping in conversation history 

with_message_history= RunnableWithMessageHistory(
    chain ,
     get_session_history ,
      input_messages_key="messages" 
      )
config={"configurable" : {"session_id": "chat5"}}